In [0]:
%sql
-- Drop existing 'bronze' schema and all its tables if they exist
DROP SCHEMA IF EXISTS workspace.bronze CASCADE;

In [0]:
%sql
-- Recreate the 'bronze' schema inside the 'workspace' Unity Catalog
CREATE SCHEMA IF NOT EXISTS workspace.bronze
COMMENT 'Bronze Layer: Raw and incremental ingestion from S3';

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Catalog and Schema Configuration
CATALOG_NAME = "workspace"
SCHEMA_NAME = "bronze"

# S3 bucket paths via External Location
S3_BUCKET_BASE = "s3://dataheib-s3-bucket"

# Source paths
S3_RAW_HEADER_PATH = f"{S3_BUCKET_BASE}/raw/"
S3_RAW_ITEM_PATH   = f"{S3_BUCKET_BASE}/raw/"

# Independent paths for Auto Loader metadata (Checkpoints and Schema Inference)
CHECKPOINT_HEADER_PATH = f"{S3_BUCKET_BASE}/_checkpoints/bronze/sales_order_header"
SCHEMA_HEADER_PATH     = f"{S3_BUCKET_BASE}/_schema_meta/bronze/sales_order_header"

CHECKPOINT_ITEM_PATH   = f"{S3_BUCKET_BASE}/_checkpoints/bronze/sales_order_item"
SCHEMA_ITEM_PATH       = f"{S3_BUCKET_BASE}/_schema_meta/bronze/sales_order_item"

In [0]:
# Ingestion Stream Configuration for Header
df_header_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", SCHEMA_HEADER_PATH)
    .option("pathGlobFilter", "sales_order_*.parquet")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(S3_RAW_HEADER_PATH)
)

# Technical Metadata Enrichment (Unity Catalog)
df_header_bronze = (
    df_header_raw
    .withColumn("_input_file_name", col("_metadata.file_path"))
    .withColumn("_ingestion_timestamp", current_timestamp())
)

# Write to Bronze Delta Table
query_header = (
    df_header_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_HEADER_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.sales_order_header")
)

query_header.awaitTermination()
print("Sales Order Header load to Bronze completed successfully.")

In [0]:
# Ingestion Stream Configuration for Item
df_item_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", SCHEMA_ITEM_PATH)
    .option("pathGlobFilter", "*sales_order_item_*.parquet")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(S3_RAW_ITEM_PATH)
)

# Technical Metadata Enrichment (Unity Catalog)
df_item_bronze = (
    df_item_raw
    .withColumn("_input_file_name", col("_metadata.file_path"))
    .withColumn("_ingestion_timestamp", current_timestamp())
)

# Write to Items Delta Table
query_item = (
    df_item_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_ITEM_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.sales_order_item")
)

query_item.awaitTermination()
print("Sales Order Item load to Bronze completed successfully.")

In [0]:
%sql
SELECT 
    COUNT(1) AS total_headers,
    MAX(_ingestion_timestamp) AS last_ingestion,
    COUNT(DISTINCT _input_file_name) AS total_processed_files
FROM workspace.bronze.sales_order_header;